In [1]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.dates as mdates
from datetime import datetime, timedelta

# --- MOCK CLASSES (Replace these with: from engine import MatchingEngine, Order, OrderType) ---
# I'm including these so the script runs out-of-the-box for demonstration.
# Delete this block when linking to your actual code.
class MockMatchingEngine:
    def __init__(self):
        self.tape = [] # List of dicts: {'time': t, 'price': p, 'qty': q}
        self.time_counter = datetime(2025, 1, 1, 9, 30)
    
    def place_order(self, order):
        # Simulation of a random match for testing visualization
        self.time_counter += timedelta(seconds=np.random.randint(1, 10))
        if random.random() > 0.7: # 30% chance of trade
            trade_price = order['price']
            trade_qty = random.randint(1, order['qty'])
            self.tape.append({
                'timestamp': self.time_counter,
                'price': trade_price,
                'qty': trade_qty
            })

class MockOrder:
    def __init__(self, side, price, qty):
        self.side = side
        self.price = price
        self.qty = qty
# -----------------------------------------------------------------------------------------

# Configuration
SEED = 42
NUM_ORDERS = 1000
OUTPUT_FILENAME = "simulation_report.pdf"

def generate_random_order():
    """Generates a random limit order for simulation."""
    side = random.choice(['buy', 'sell'])
    # Random walk-ish price generation
    base_price = 100.0
    price_noise = np.random.normal(0, 2.0)
    price = round(base_price + price_noise, 2)
    qty = random.randint(1, 100)
    
    # Return a dict or your actual Order object
    # If using your class: return Order(side=side, price=price, qty=qty)
    return {'side': side, 'price': price, 'qty': qty}

def run_simulation():
    print(f"--- Starting Simulation (Seed: {SEED}) ---")
    
    # 1. Seed RNG for reproducibility
    random.seed(SEED)
    np.random.seed(SEED)
    
    # 2. Initialize Engine
    # engine = MatchingEngine()  <-- Use your actual class here
    engine = MockMatchingEngine() 
    
    print(f"Generating and processing {NUM_ORDERS} orders...")
    
    # 3. Run Event Loop
    for i in range(NUM_ORDERS):
        order = generate_random_order()
        engine.place_order(order)
        
        # Optional: Print progress every 100 orders
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1} orders...")

    print("Simulation complete.")
    return engine.tape

def process_data(tape):
    """Converts raw trade tape to OHLC DataFrame."""
    if not tape:
        print("WARNING: No trades occurred! Check your matching logic.")
        return pd.DataFrame()

    print("Building DataFrames...")
    df_tape = pd.DataFrame(tape)
    
    # Ensure timestamp is datetime index
    df_tape['timestamp'] = pd.to_datetime(df_tape['timestamp'])
    df_tape.set_index('timestamp', inplace=True)
    
    # 4. Resample Tape -> OHLC
    # We resample to 1-minute bars ('1min')
    ohlc = df_tape['price'].resample('1min').ohlc()
    
    # Calculate Volume
    volume = df_tape['qty'].resample('1min').sum()
    ohlc['volume'] = volume
    
    # Drop empty intervals (no trades in that minute)
    ohlc.dropna(inplace=True)
    
    return ohlc

def generate_report(ohlc_data):
    """Generates a PDF report with visualizations."""
    print(f"Generating report: {OUTPUT_FILENAME}...")
    
    with PdfPages(OUTPUT_FILENAME) as pdf:
        # --- Page 1: Candlestick Chart ---
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Plotting manually for control (or use mplfinance if installed)
        # Up candles (Close >= Open)
        up = ohlc_data[ohlc_data.close >= ohlc_data.open]
        # Down candles (Close < Open)
        down = ohlc_data[ohlc_data.close < ohlc_data.open]
        
        # Plot Up candles (Green)
        ax.bar(up.index, up.close - up.open, bottom=up.open, width=0.0005, color='green', label='Up')
        ax.vlines(up.index, up.low, up.high, color='green', linewidth=1)
        
        # Plot Down candles (Red)
        ax.bar(down.index, down.close - down.open, bottom=down.open, width=0.0005, color='red', label='Down')
        ax.vlines(down.index, down.low, down.high, color='red', linewidth=1)
        
        ax.set_title("Simulation Run: Price Action (1-Min OHLC)")
        ax.set_ylabel("Price")
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        plt.grid(True, alpha=0.3)
        
        pdf.savefig(fig)
        plt.close()
        
        # --- Page 2: Volume Profile ---
        fig2, ax2 = plt.subplots(figsize=(10, 6))
        ax2.bar(ohlc_data.index, ohlc_data['volume'], width=0.0005, color='blue', alpha=0.6)
        ax2.set_title("Simulation Run: Volume Profile")
        ax2.set_ylabel("Volume")
        ax2.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
        
        pdf.savefig(fig2)
        plt.close()
        
    print("Report generated successfully.")

if __name__ == "__main__":
    # 1. Run the simulation loop
    raw_tape = run_simulation()
    
    # 2. Process data
    if raw_tape:
        df_ohlc = process_data(raw_tape)
        
        # 3. Generate PDF
        if not df_ohlc.empty:
            generate_report(df_ohlc)
            print(f"Done. Check {OUTPUT_FILENAME}")
        else:
            print("No OHLC data generated.")

--- Starting Simulation (Seed: 42) ---
Generating and processing 1000 orders...
Processed 100 orders...
Processed 200 orders...
Processed 300 orders...
Processed 400 orders...
Processed 500 orders...
Processed 600 orders...
Processed 700 orders...
Processed 800 orders...
Processed 900 orders...
Processed 1000 orders...
Simulation complete.
Building DataFrames...
Generating report: simulation_report.pdf...
Report generated successfully.
Done. Check simulation_report.pdf
